In [ ]:
import json 
import openpyxl 
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side 
from openpyxl.utils import get_column_letter

def employee_report(json_file, excel_file):
    with open(json_file, 'r', encoding ='utf-8') as f:
        data = json.load(f)

    company = data.get('company', '')
    department = data.get('department', '')
    last_updated = data.get('last_updated' ,'')
    employees = data.get('employees', [])
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Employee Directory"

    thin_border = Border(
        left = Side(style='thin'),
        right = Side(style ='thin'),
        top = Side(style= 'thin'),
        bottom = Side(style= 'thin')
    )

    fill_lead = PatternFill(start_color='FFD700', end_color= 'FFD700', fill_type= 'solid')
    fill_senior = PatternFill(start_color='B4C7E7', end_color= 'B4C7E7', fill_type= 'solid')
    fill_gray = PatternFill(start_color='E7E6E6', end_color= 'E7E6E6', fill_type= 'solid')
    ws.merge_cells("A1:H1")
    ws["A1"] = f"{company} - {department} Department" 
    ws["A1"].font = Font(size = 14, bold = True, color = "FFFFFF")
    ws["A1"].fill = PatternFill(start_color= "4472C4", end_color= "4472C4" , fill_type= "solid")
    ws["A1"].alignment = Alignment( horizontal = "center", vertical = "center")
    ws.merge_cells("A2:H2")
    ws["A2"] = f"Last updated: {last_updated} "
    ws["A2"].font = Font(italic = True)    
    ws["A2"].alignment = Alignment(horizontal = "center", vertical = "center")

    headers = ["Employee_ID", "Name", "Position", "Email", "Hire_Date", "Salary", "Years Of Service", "Skills"]
    ws.append([])
    ws.append(headers)
    
    for col_idx in range(1,9):
        cell = ws.cell(row = 4, column = col_idx)
        cell.font = Font(bold = True, color ="FFFFFF")
        cell.fill = PatternFill(start_color= "203864", end_color= "203864", fill_type = "solid")
        cell.alignment = Alignment(horizontal = "center", vertical = "center")

    total_salary = 0
    start_row = 5
    
    if '/' in last_updated:
      current_year = int(last_updated.split('/')[-1])
    elif '-' in last_updated:
      current_year = int(last_updated.split('-')[0])
    else:
      current_year = 2024

    for emp in employees:
      # חילוץ שנה תקין מתאריך קליטת העובד
      h_date = emp.get('hire_date', '')
      if '/' in h_date:
        hire_year = int(h_date.split('/')[-1])
      elif '-' in h_date:
        hire_year = int(h_date.split('-')[0])
      else:
        hire_year = current_year
    for emp in employees:
            hire_year = int(emp['hire_date'].split('-')[0])
            years_of_Service = current_year - hire_year
            skills_str = ", ".join(emp.get('skills', [])) 
            salary = emp.get('salary', 0)
            total_salary += salary

            row_data = [
                emp.get('employee_id', ''),
                emp.get('name', ''),
                emp.get('position',''),
                emp.get('email',''),
                emp.get('hire_date',''),
                salary,
                years_of_Service,
                skills_str,
            ]      
            ws.append(row_data)
            curr_row = ws.max_row
            ws.cell(row = curr_row, column = 6).number_format = '$#,##0'
            pos = emp.get('position', '')
            row_fill = None
            if "TechLead" in pos or "Tech Lead" in pos or "Lead" in pos:
                row_fill = fill_lead
            elif "Senior" in pos or "Sr" in pos:
                row_fill = fill_senior

            if row_fill:
                for c in range(1,9):
                    ws.cell(row = curr_row , column = c).fill = row_fill
    summary_row = ws.max_row + 1
    total_employees = len(employees)
    avg_salary = (total_salary / total_employees) if total_employees > 0 else 0

    ws.merge_cells(start_row= summary_row, start_column = 1, end_row = summary_row, end_column = 5)
    cell_total = ws.cell(row= summary_row, column = 1, value =f"Total employees: {total_employees} ")
    cell_total.font = Font(bold =True)
    cell_total.alignment = Alignment (horizontal = "right", vertical = "center")

    cell_avg = ws.cell(row = summary_row, column = 6, value = avg_salary) 
    cell_avg.font = Font(bold = True)
    cell_avg.number_format = '$#,##0'

    ws.merge_cells(start_row = summary_row, start_column= 7, end_row= summary_row, end_column = 8)
    cell_label = ws.cell(row = summary_row, column  = 7, value = "Average Salary")
    cell_label.font = Font(bold = True)
    cell_label.alignment = Alignment(horizontal = "right", vertical = "center")
    cell_label.number_format ='$#,##0'

    for c in range (1, 9):
            ws.cell(row = summary_row, column = c).fill = fill_gray

    for r in range(4, summary_row +1):
            for c in range(1, 9):
                ws.cell(row = r ,column =c).border = thin_border
            
    for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            col_letter = get_column_letter(col[0].column)
            ws.column_dimensions[col_letter].width = max(max_len + 4, 12)
    wb.save(excel_file)
    print(f"כמות עובדים:{total_employees}")
    print(f"שכר ממוצע:{avg_salary:,.2f}")
employee_report("employees.json", "employee_directory.xlsx")

In [ ]:
def mystery_Calc(val , step , origin):
    if step > 3:
        return val
    if step == 1:
        return mystery_Calc(val *3 , step +1 , origin)
    if step == 2:
        return mystery_Calc(val +12 , step +1 , origin)
    if step == 3:
        return mystery_Calc(val- origin, step+1, origin)
num = int(input("think of any number: "))   
target_num = mystery_Calc(num, 1, num)
print(f"the initial number is : {num}\nthe finished number is : {target_num} " )


the initial number is : 1
the finished number is : 14 


In [ ]:
def merger_sorted_lists(list1, list2):
    merged = []
    i = 0 
    j = 0
    
    while i < len(list1) and j < len(list2):
        if list1[i] < list2[j]:
            merged.append(list1[i])
            i += 1
        elif list2[j] < list1[i]:
            merged.append(list2[j])
            j += 1
        else:
            merged.append(list1[i])
            i += 1 
            j += 1
            
    while i < len(list1):
        merged.append(list1[i])
        i += 1            
        
    while j < len(list2):
        merged.append(list2[j])
        j += 1
        
    return merged